# Part 3b — Learning by Doing: production-based learning

### Two learning channels, and why one of them breaks SOS2

Part 3 made capital cost fall with cumulative **installed capacity**. That is *learning by
building*, and it is real — construction know-how and equipment supply chains genuinely improve.
But Wright's original law was about cumulative **production**, and the distinction is not
cosmetic: a plant that is built and then idled learns nothing.

This notebook keeps both channels and puts each where it belongs:

| Channel | Driver | Applies to | Formulation |
|---|---|---|---|
| **A — learning by building** | cumulative **capacity** | capex | SOS2 (exact, from Part 3) |
| **B — learning by doing** | cumulative **production**, lagged | opex | disjunctive tiers |

Since operating cost is roughly 65% of the objective against capex's 20%, **channel B is where
the money is** — and it is also the harder one to formulate.

### Why Part 3's capacity-based learning is a legitimate bound

Learning by capacity grants cost reductions from building alone; learning by production requires
building *and* running. So the capacity version can never cost more, which makes Part 3 a valid
**lower bound** and the gap between them a measurable quantity — not merely a different answer.

### What this notebook adds

- Cumulative production tracked by stage, region and period — **undiscounted**
- Opex learning via lagged disjunctive tiers, with thresholds calibrated from a first solve
- **Utilization** diagnostics, capacity-year weighted
- **Local content minimums** (a government lever) with one-sided undersupply deviation
- **Overproduction with disposal cost** — scaffolding for Part 4, verified inactive here
- Two questions answered by experiment: does production learning change the *plan*, and is
  pump-and-dump ever profitable?

## Formulation: what is new relative to Part 3

### New state variable

$$\text{cumprod}_{s,\kappa,p} \;=\; \sum_{r \in \kappa} \sum_{q \le p} L_q \sum_{v} x_{s,r,v,q}$$

where $\kappa$ is the learning **scope** — the region itself if learning is regional, or all
regions pooled if it spills over. Note $L_q$, the **period length in years, undiscounted**.
Cumulative production is a count of physical units; atoms do not discount. Using $\omega_q$ here
would be a silent error.

### Why this cannot be SOS2

Operating cost would be

$$\underbrace{U(\text{cumprod})}_{\text{variable}} \times \underbrace{x}_{\text{variable}}$$

which is **bilinear** — nonconvex, outside SOS2's reach. In Part 3 the curve's argument was the
same quantity being paid for, so the interpolated cumulative cost entered the objective directly
and no product of variables appeared.

### Disjunctive tiers

$$\sum_j z_{s,\kappa,p,j} = 1, \qquad z \in \{0,1\}$$
$$\text{cumprod}_{s,\kappa,p-\text{lag}} \ge T_{s,j-1} - M(1 - z_{s,\kappa,p,j})$$
$$\text{cumprod}_{s,\kappa,p-\text{lag}} \le T_{s,j} + M(1 - z_{s,\kappa,p,j})$$
$$\sum_j \chi_{s,r,p,j} = \sum_v x_{s,r,v,p}, \qquad
\chi_{s,r,p,j} \le \overline{x}\, z_{s,\kappa,p,j}$$
$$\text{opex} = \sum_p \omega_p \sum_j o_s\, m_{s,j}\, \chi_{s,r,p,j}$$

Every product is now (constant × variable).

**The lag is defined in years, not periods.** With periods of 1, 3, 5 and 9 years, "lagged one
period" would mean one year early and nine years late — learning would artificially decelerate as
periods coarsen, and it would look like a finding.

**Thresholds are placed as doublings**, so the tier multiplier is exactly Wright's law: each
doubling of cumulative production multiplies operating cost by $(1-LR)$.

### New variables and constraints

| Symbol | Meaning |
|---|---|
| $u_{r,p}$ | unmet final demand — penalty $\pi^{short}$ |
| $d_{s,r,p}$ | **one-sided** undersupply of a local content minimum — penalty $\pi^{dev}$ |
| $g_{r,p}$ | overproduction sent to disposal — cost $\pi^{disp}$ |

Demand becomes an equality with two escape valves:
$$\sum_{r_1} f_{\text{MFG},r_1,r,p} + u_{r,p} - g_{r,p} = D_{r,p}$$

Local content is one-sided, because exceeding a minimum is not a violation:
$$\sum_v x_{s,r,v,p} + d_{s,r,p} \;\ge\; \text{MIN}_{s,r,p}$$

## 1. Setup

Roughly 1,600 variables at the largest — under the `pip` licence cap of 2,000. The period
structure is slightly shorter than Part 3 (37 years in 13 periods) to make room for the tier
binaries.

In [ ]:
!pip install gurobipy --quiet
import math
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})
print("gurobipy", gp.gurobi.version())

## 2. Sets, time, and the year-to-period map

`YEAR_TO_P` is what lets the learning lag be specified in years and then mapped onto whichever
period contains that year.

In [ ]:
# ================= SETS =================
REGIONS = ['R1', 'R2']
STAGES  = ['MINE', 'PROC', 'MFG']
NODES   = [(s, r) for s in STAGES for r in REGIONS]
ARCS    = [(s, r1, r2) for s in STAGES for r1 in REGIONS for r2 in REGIONS]
TRANSPORT = {(r1, r2): (0.5 if r1 == r2 else 2.0) for r1 in REGIONS for r2 in REGIONS}

# ================= TIME =================
BLOCKS = [(6, 1), (4, 3), (2, 5), (1, 9)]
LEN, START = [], []
_y = 1
for _c, _L in BLOCKS:
    for _ in range(_c):
        LEN.append(_L); START.append(_y); _y += _L
P       = list(range(len(LEN)))
HORIZON = _y - 1
YEARS   = {p: list(range(START[p], START[p] + LEN[p])) for p in P}
DR      = 0.05
OMEGA   = {p: sum(1 / (1 + DR) ** t for t in YEARS[p]) for p in P}   # money weight
REPORT_UNTIL = 28

# map a calendar year to the period containing it (for the year-based learning lag)
YEAR_TO_P = {t: p for p in P for t in YEARS[p]}

In [ ]:
pd.DataFrame([dict(period=p, years=f"{START[p]}-{START[p]+LEN[p]-1}", length_years=LEN[p],
                   omega_money=round(OMEGA[p], 4), L_physical=LEN[p])
              for p in P])

Two weight columns, deliberately side by side. `omega_money` discounts and is used for every cost
term; `L_physical` does not discount and is used for cumulative production. Conflating them is the
classic variable-period bug.

## 3. Technology, cost, legacy and demand

Unchanged from Part 3. Reproduced so this notebook stands alone.

In [ ]:
# ================= TECH / COST =================
LIFE = 25
LEAD = {'MINE': 1, 'PROC': 2, 'MFG': 2}
CAP_MIN, CAP_MAX = 60.0, 260.0
FIXED   = {'MINE': 900.0, 'PROC': 1500.0, 'MFG': 1300.0}
UNIT    = {'MINE': 7.0,   'PROC': 11.0,   'MFG': 9.5}
OPERATE = {'MINE': 1.2,   'PROC': 2.0,    'MFG': 2.4}
CRF = DR * (1 + DR) ** LIFE / ((1 + DR) ** LIFE - 1)
ONLINE = {(s, p): START[p] + LEAD[s] for s in STAGES for p in P}
MU = {(s, v): CRF * sum(1 / (1 + DR) ** t
                        for t in range(ONLINE[s, v], ONLINE[s, v] + LIFE) if t <= HORIZON)
      for s in STAGES for v in P}

# ================= LEGACY =================
LEGACY_CAP = {('MINE','R1'):200, ('MINE','R2'):180, ('PROC','R1'):170,
              ('PROC','R2'):150, ('MFG','R1'):130,  ('MFG','R2'):120}
LEGACY_RET = {('MINE','R1'):9, ('MINE','R2'):14, ('PROC','R1'):12,
              ('PROC','R2'):19, ('MFG','R1'):16,  ('MFG','R2'):24}
LEGACY_BYR = -8

## 4. Vintage efficiency (yield)

Unchanged from Part 3: $\eta$ is a **yield** in the constraint matrix, indexed by vintage and
operating period, with a frontier channel ($\alpha$) and a within-life channel ($\beta < \alpha$).

In [ ]:
# ================= EFFICIENCY (yield) =================
ETA_CEIL = {'MINE':0.92, 'PROC':0.95, 'MFG':0.93}
ETA_BASE = {'MINE':0.86, 'PROC':0.80, 'MFG':0.78}
ALPHA    = {'MINE':0.000,'PROC':0.030,'MFG':0.025}
BETA     = {'MINE':0.000,'PROC':0.010,'MFG':0.008}
DELTA_BAR= {'MINE':0.02, 'PROC':0.05, 'MFG':0.05}
ETA_FLOOR= 0.60
VINTAGES = [-1] + P
BYEAR    = {v: (LEGACY_BYR if v == -1 else START[v]) for v in VINTAGES}
ETA = {}
for s in STAGES:
    for v in VINTAGES:
        fr = ETA_CEIL[s] - (ETA_CEIL[s] - ETA_BASE[s]) * (1 - ALPHA[s]) ** (BYEAR[v] - 1)
        fr = max(ETA_FLOOR, min(fr, ETA_CEIL[s]))
        for p in P:
            age = max(0, START[p] - BYEAR[v])
            aged = ETA_CEIL[s] - (ETA_CEIL[s] - fr) * (1 - BETA[s]) ** age
            ETA[s, v, p] = max(ETA_FLOOR, min(fr + DELTA_BAR[s], aged))

# ================= DEMAND =================
DEMAND = {}
for r, base, g in [('R1', 100.0, 0.008), ('R2', 70.0, 0.026)]:
    for p in P:
        DEMAND[r, p] = sum(base * (1 + g) ** (t - 1) for t in YEARS[p]) / LEN[p]

## 5. Active sets

In [ ]:
# ================= ACTIVE SETS =================
ACTIVE = [(s, r, v, p) for (s, r) in NODES for v in VINTAGES for p in P
          if (v == -1 and START[p] <= LEGACY_RET[s, r])
          or (v >= 0 and ONLINE[s, v] <= START[p] <= ONLINE[s, v] + LIFE - 1)]
VIN = {(s, r, p): [v for (ss, rr, v, pp) in ACTIVE if (ss, rr, pp) == (s, r, p)]
       for (s, r) in NODES for p in P}
BUILD = [(s, r, v) for (s, r) in NODES for v in P if ONLINE[s, v] <= HORIZON]

## 6. Channel A — capex learning on cumulative capacity (SOS2)

Same construction as Part 3: linearize **cumulative** capex against cumulative capacity, charge
the period increment, enforce SOS2 so the relaxation cannot ride the chord below a concave curve.

In [ ]:
# ================= CHANNEL A: capex learning (capacity, SOS2) =================
LEARN_STAGES = ['PROC', 'MFG']
LR_CAPEX = 0.15
Q_START, Q_ADD = 400.0, 1000.0
CAPEX_FLOOR = 0.60
NBP = 9
_bc = -math.log2(1 - LR_CAPEX)
_U0 = sum(UNIT[s] for s in LEARN_STAGES) / len(LEARN_STAGES)

def capex_unit(q):
    return max(CAPEX_FLOOR * _U0, _U0 * (q / Q_START) ** (-_bc))

def capex_cum(q, n=600):
    if q <= Q_START:
        return 0.0
    h = (q - Q_START) / n
    return sum(0.5 * (capex_unit(Q_START + i*h) + capex_unit(Q_START + (i+1)*h)) * h
               for i in range(n))

K   = list(range(NBP))
QBP = [Q_START + Q_ADD * k / (NBP - 1) for k in K]
CBP = [capex_cum(q) for q in QBP]
MU_TECH = {p: MU['PROC', p] for p in P}

## 7. Channel B — opex learning parameters

`TIER_Q` and `TIER_M` are deliberately left **empty** here. Thresholds chosen a priori usually
either never activate or all activate immediately; §9 calibrates them from a solve.

`LEARN_SCOPE` decides whether know-how stays inside a region or spills across both. That switch
matters more than it looks: **regional** learning makes pre-emption pay, because getting to a tier
first is a durable advantage. **Global** learning creates free-riding, because your rival's output
lowers your costs too. Part 4 will lean on this.

In [ ]:
# ================= CHANNEL B: opex learning (production, lagged tiers) =================
LR_OPEX   = 0.18            # opex fall per doubling of cumulative production
OPEX_FLOOR= 0.65            # floor as fraction of base opex
LAG_YEARS = 3               # know-how embodies with a delay, defined in YEARS
N_TIERS   = 3               # tier 0 = no discount
LEARN_SCOPE = 'regional'    # 'regional' | 'global'

# tier thresholds are calibrated from a no-learning solve (see calibrate_tiers)
TIER_Q = {}                 # (stage) -> [t1, t2, t3]  cumulative production thresholds
TIER_M = {}                 # (stage) -> [m0, m1, m2, m3] opex multipliers

# ================= NON-MARKET LEVERS (government) =================
TIER_MIN = {}               # (stage, region, period) -> minimum throughput (local content)
TIER_MIN_PHASE_IN = 6       # no minimum binds before this year

# ================= PENALTIES =================
PEN_SHORT   = 90.0          # unmet final demand
PEN_DEVIATE = 35.0          # under-supply of a tier minimum (ONE-SIDED)
PEN_DISPOSE = 12.0          # overproduction disposal

In [ ]:
def set_tiers(prod_by_stage):
    """Thresholds placed as DOUBLINGS of cumulative production, so the multiplier
    per tier is exactly Wright's law: each doubling multiplies opex by (1 - LR)."""
    for s in STAGES:
        top = max(prod_by_stage[s], 1.0)
        q1 = top / 8.0
        TIER_Q[s] = [q1 * 2 ** j for j in range(N_TIERS - 1)]      # q1, 2q1, ...
        TIER_M[s] = [max(OPEX_FLOOR, (1 - LR_OPEX) ** j) for j in range(N_TIERS)]

In [ ]:
def set_tier_minimums(level=0.0):
    """Government local-content lever: min throughput at each stage/region."""
    TIER_MIN.clear()
    if level <= 0:
        return
    for (s, r) in NODES:
        for p in P:
            TIER_MIN[s, r, p] = 0.0 if START[p] < TIER_MIN_PHASE_IN else level

## 8. The model

One builder, because this notebook deliberately solves four learning variants and several
penalty settings. Inside, the constraints are written the same flat way as Part 3 — `addConstrs`
with generator expressions, `tupledict.sum()` wildcards.

Reading order inside `build_model`: variables, sizing, capacity, network balance, government
levers, cumulative production state, channel A, channel B, objective.

In [ ]:
def build_model(learning='production', allow_dispose=True, quiet=True, mipgap=0.005,
                pen_dispose=None, pen_deviate=None):
    """learning: 'none' | 'capacity' | 'production' | 'both'"""
    pd_ = PEN_DISPOSE if pen_dispose is None else pen_dispose
    pv_ = PEN_DEVIATE if pen_deviate is None else pen_deviate
    m = gp.Model()
    m.Params.OutputFlag = 0 if quiet else 1
    m.Params.MIPGap = mipgap

    build = m.addVars(BUILD, vtype=GRB.BINARY, name='build')
    size  = m.addVars(BUILD, lb=0.0, ub=CAP_MAX, name='size')
    thr   = m.addVars(ACTIVE, lb=0.0, name='thr')
    flow  = m.addVars(ARCS, P, lb=0.0, name='flow')
    short = m.addVars(REGIONS, P, lb=0.0, name='short')
    dev   = m.addVars(NODES, P, lb=0.0, name='dev')       # one-sided undersupply
    disp  = m.addVars(REGIONS, P, lb=0.0, name='disp')    # overproduction disposal

    m.addConstrs((size[s,r,v] <= CAP_MAX*build[s,r,v] for (s,r,v) in BUILD), name='size_ub')
    m.addConstrs((size[s,r,v] >= CAP_MIN*build[s,r,v] for (s,r,v) in BUILD), name='size_lb')
    m.addConstrs((thr[s,r,v,p] <= (LEGACY_CAP[s,r] if v == -1 else size[s,r,v])
                  for (s,r,v,p) in ACTIVE), name='cap')
    m.addConstrs((gp.quicksum(ETA[s,v,p]*thr[s,r,v,p] for v in VIN[s,r,p])
                  == flow.sum(s,r,'*',p) for (s,r) in NODES for p in P), name='node_out')
    m.addConstrs((flow.sum('MINE','*',r,p)
                  == gp.quicksum(thr['PROC',r,v,p] for v in VIN['PROC',r,p])
                  for r in REGIONS for p in P), name='in_proc')
    m.addConstrs((flow.sum('PROC','*',r,p)
                  == gp.quicksum(thr['MFG',r,v,p] for v in VIN['MFG',r,p])
                  for r in REGIONS for p in P), name='in_mfg')
    # demand: deliveries cover demand; surplus goes to disposal
    m.addConstrs((flow.sum('MFG','*',r,p) + short[r,p] - disp[r,p] == DEMAND[r,p]
                  for r in REGIONS for p in P), name='demand')
    if not allow_dispose:
        m.addConstrs((disp[r,p] == 0 for r in REGIONS for p in P), name='no_disp')

    # government local-content minimums, one-sided deviation
    if TIER_MIN:
        m.addConstrs((gp.quicksum(thr[s,r,v,p] for v in VIN[s,r,p]) + dev[s,r,p]
                      >= TIER_MIN.get((s,r,p), 0.0)
                      for (s,r) in NODES for p in P), name='local_content')

    # ---------- cumulative production state (undiscounted: uses LEN, not OMEGA) ----------
    if LEARN_SCOPE == 'regional':
        SCOPE = [(s_, r_) for (s_, r_) in NODES]
    else:
        SCOPE = [(s_, 'ALL') for s_ in STAGES]
    CUMPROD_UB = 3.0 * CAP_MAX * HORIZON * (len(REGIONS) if LEARN_SCOPE == 'global' else 1)
    cumprod = m.addVars(SCOPE, P, lb=0.0, ub=CUMPROD_UB, name='cumprod')
    m.addConstrs((cumprod[s_, rk, p] ==
                  gp.quicksum(LEN[q] * thr[s_, r_, v, q]
                              for r_ in (REGIONS if rk == 'ALL' else [rk])
                              for q in P if q <= p
                              for v in VIN[s_, r_, q])
                  for (s_, rk) in SCOPE for p in P), name='cum_prod')

    # ---------- CHANNEL A: capex learning on cumulative capacity (SOS2) ----------
    capex = gp.quicksum(MU[s,v]*FIXED[s]*build[s,r,v] for (s,r,v) in BUILD) \
          + gp.quicksum(MU[s,v]*UNIT[s]*size[s,r,v]
                        for (s,r,v) in BUILD if s not in LEARN_STAGES)
    if learning in ('capacity', 'both'):
        Q = m.addVars(P, lb=Q_START, ub=Q_START+Q_ADD, name='Qcum')
        Cc = m.addVars(P, lb=0.0, name='Ccum')
        lam = m.addVars(P, K, lb=0.0, ub=1.0, name='lam')
        m.addConstrs((lam.sum(p,'*') == 1 for p in P), name='sos_cvx')
        m.addConstrs((Q[p] == gp.quicksum(QBP[k]*lam[p,k] for k in K) for p in P), name='sosQ')
        m.addConstrs((Cc[p] == gp.quicksum(CBP[k]*lam[p,k] for k in K) for p in P), name='sosC')
        m.addConstrs((Q[p] == Q_START + gp.quicksum(size[s,r,v] for (s,r,v) in BUILD
                                                    if s in LEARN_STAGES and v <= p)
                      for p in P), name='cumcap')
        for p in P:
            m.addSOS(GRB.SOS_TYPE2, [lam[p,k] for k in K])
        capex += gp.quicksum(MU_TECH[p]*(Cc[p] - (Cc[p-1] if p > 0 else 0.0)) for p in P)
    else:
        capex += gp.quicksum(MU[s,v]*UNIT[s]*size[s,r,v]
                             for (s,r,v) in BUILD if s in LEARN_STAGES)

    # ---------- CHANNEL B: opex learning on LAGGED cumulative production (tiers) ----------
    if learning in ('production', 'both') and TIER_Q:
        J = list(range(N_TIERS))
        z = m.addVars(SCOPE, P, J, vtype=GRB.BINARY, name='tier')
        m.addConstrs((z.sum(s_, rk, p, '*') == 1 for (s_, rk) in SCOPE for p in P),
                     name='one_tier')
        # the lag is defined in YEARS, then mapped to whichever period holds that year,
        # so it does not silently stretch as the periods coarsen
        LAGP = {p: YEAR_TO_P[max(1, START[p] - LAG_YEARS)] for p in P}
        BIGQ = CUMPROD_UB
        m.addConstrs((cumprod[s_, rk, LAGP[p]] >= TIER_Q[s_][j-1] - BIGQ*(1 - z[s_,rk,p,j])
                      for (s_, rk) in SCOPE for p in P for j in J if j > 0),
                     name='tier_floor')
        m.addConstrs((cumprod[s_, rk, LAGP[p]] <= TIER_Q[s_][j] + BIGQ*(1 - z[s_,rk,p,j])
                      for (s_, rk) in SCOPE for p in P for j in J if j < N_TIERS-1),
                     name='tier_ceil')
        # Disaggregate throughput across tiers so (multiplier x throughput) is LINEAR.
        # The opex RATE does not depend on vintage, so splitting NODE-level throughput
        # is exactly equivalent to splitting vintage-level -- and far smaller.
        tsplit = m.addVars(NODES, P, J, lb=0.0, name='tsplit')
        m.addConstrs((tsplit.sum(s_, r_, p, '*')
                      == gp.quicksum(thr[s_, r_, v, p] for v in VIN[s_, r_, p])
                      for (s_, r_) in NODES for p in P), name='tsplit_sum')
        m.addConstrs((tsplit[s_, r_, p, j]
                      <= 3 * CAP_MAX * z[s_, (r_ if LEARN_SCOPE == 'regional' else 'ALL'), p, j]
                      for (s_, r_) in NODES for p in P for j in J), name='tier_link')
        operate = gp.quicksum(OMEGA[p]*OPERATE[s_]*TIER_M[s_][j]*tsplit[s_,r_,p,j]
                              for (s_,r_) in NODES for p in P for j in J)
        m._z = z
    else:
        operate = gp.quicksum(OMEGA[p]*OPERATE[s]*thr[s,r,v,p] for (s,r,v,p) in ACTIVE)
        m._z = None

    transport = gp.quicksum(OMEGA[p]*TRANSPORT[r1,r2]*flow[s,r1,r2,p]
                            for (s,r1,r2) in ARCS for p in P)
    penalty = gp.quicksum(OMEGA[p]*PEN_SHORT*short[r,p] for r in REGIONS for p in P) \
            + gp.quicksum(OMEGA[p]*pv_*dev[s,r,p] for (s,r) in NODES for p in P) \
            + gp.quicksum(OMEGA[p]*pd_*disp[r,p] for r in REGIONS for p in P)

    m.setObjective(capex + operate + transport + penalty, GRB.MINIMIZE)
    m._e = dict(capex=capex, operate=operate, transport=transport, penalty=penalty)
    m._v = dict(build=build, size=size, thr=thr, flow=flow, short=short,
                dev=dev, disp=disp, cumprod=cumprod)
    m._scope = SCOPE
    return m

## 9. Calibrating the tiers

Solve with **no learning**, read the cumulative production that actually occurs, and place
thresholds across that range as doublings. This is the same discipline as re-meshing the SOS2
breakpoints in Part 3: a threshold outside the realised range is a dead binary, and learning would
silently never activate.

In [ ]:
def calibrate_tiers(mipgap=0.005):
    """Solve with NO learning, read realised cumulative production, place thresholds
    across that range. Thresholds chosen a priori often never activate."""
    m = build_model(learning='none', mipgap=mipgap)
    m.optimize()
    thr = m._v['thr']
    prod = {}
    for s_ in STAGES:
        per_region = {rr: sum(LEN[p]*thr[ss,r,v,p].X for (ss,r,v,p) in ACTIVE
                              if ss == s_ and r == rr) for rr in REGIONS}
        prod[s_] = (max(per_region.values()) if LEARN_SCOPE == 'regional'
                    else sum(per_region.values()))
    set_tiers(prod)
    return m.ObjVal, prod

In [ ]:
def utilization(m):
    """Capacity-year weighted utilization by stage and region."""
    v = m._v
    out = {}
    for (s, r) in NODES:
        cy = used = 0.0
        for (ss, rr, vv, p) in ACTIVE:
            if (ss, rr) != (s, r):
                continue
            cap = LEGACY_CAP[s,r] if vv == -1 else v['size'][s,r,vv].X
            cy   += cap * LEN[p]
            used += v['thr'][s,r,vv,p].X * LEN[p]
        out[s, r] = (used/cy if cy > 1e-6 else None)
    return out

In [ ]:
base_obj, prod = calibrate_tiers()
print(f"no-learning objective : {base_obj:.1f}")
print("cumulative production by stage (scope max):",
      {s: round(prod[s], 1) for s in STAGES})
print("tier thresholds :", {s: [round(q, 1) for q in TIER_Q[s]] for s in STAGES})
print("tier multipliers:", {s: [round(x, 4) for x in TIER_M[s]] for s in STAGES})
print(f"\n(multipliers are exactly (1-LR)^j with LR={LR_OPEX:.0%}, floored at {OPEX_FLOOR})")

## 10. Four learning variants

- **none** — flat costs, the loosest reference
- **capacity** — Part 3's channel A only
- **production** — channel B only
- **both** — the intended model

In [ ]:
rows = []
for mode in ['none', 'capacity', 'production', 'both']:
    m = build_model(learning=mode)
    m.optimize()
    v, e = m._v, m._e
    rows.append(dict(learning=mode, objective=round(m.ObjVal, 1),
                     capex=round(e['capex'].getValue(), 1),
                     opex=round(e['operate'].getValue(), 1),
                     builds=sum(1 for k in BUILD if v['build'][k].X > 0.5),
                     capacity=round(sum(v['size'][k].X for k in BUILD), 1),
                     disposal=round(sum(v['disp'][r, p].X for r in REGIONS for p in P), 2),
                     variables=m.NumVars, binaries=m.NumBinVars))
pd.DataFrame(rows)

Read the two cost columns rather than the total. **Channel A moves `capex` and leaves `opex`
essentially alone; channel B moves `opex` and leaves `capex` alone.** That separation is the point
of putting each channel on its own cost component — and it is a useful check that neither is
leaking into the other.

Note also that `disposal` is **exactly zero** in every variant. That is expected and it is a
verification: under cost minimization with no revenue, overproduction is strictly dominated. The
mechanism is wired and inactive. §13 tests how hard it is to make it activate.

## 11. Utilization — the check you predicted

With continuous sizing and no reason to hold idle capacity, a cost-minimizing model should run its
plants hard. If utilization came back low, something would be wrong — most likely lumpy legacy
capacity or a stage mismatch upstream.

Weighted by **capacity-years**, not averaged across periods: a naive mean over periods of 1 and 9
years would badly mislead. Utilization is undefined where capacity is zero, so those pairs are
excluded rather than counted as zero.

In [ ]:
util = []
for mode in ['none', 'production']:
    m = build_model(learning=mode); m.optimize()
    u = utilization(m)
    for (s, r), val in sorted(u.items()):
        util.append(dict(learning=mode, stage=s, region=r,
                         utilization_pct=None if val is None else round(100*val, 1)))
pd.DataFrame(util).pivot(index=['stage', 'region'], columns='learning',
                         values='utilization_pct')

High across the board, as expected. The lower figure at MFG R1 is not a defect: a legacy asset and
a new build overlap there for several periods, and the model dispatches the newer, higher-yield
vintage first — so the older one shows slack while it runs out its life. That is exactly the
behaviour vintage indexing exists to represent.

## 12. Which tiers actually activate?

A dead tier is a wasted binary and a silent failure of the learning mechanism. Print the selected
tier per stage, scope and period.

In [ ]:
m = build_model(learning='production'); m.optimize()
z, cp = m._z, m._v['cumprod']
rows = []
for (s_, rk) in m._scope:
    sel = []
    for p in P:
        j = [jj for jj in range(N_TIERS) if z[s_, rk, p, jj].X > 0.5]
        sel.append(j[0] if j else None)
    rows.append(dict(stage=s_, scope=rk, tier_by_period=sel,
                     cumprod_end=round(cp[s_, rk, P[-1]].X, 1)))
pd.DataFrame(rows)

Tiers progress 0 → 1 → 2 monotonically, which is what a cumulative driver should produce. The
three-year lag is visible: the tier changes some periods after cumulative production crosses the
threshold, because know-how takes time to embody.

## 13. Is pump-and-dump ever profitable?

The concern: with production-driven learning, a firm might overproduce and dump the surplus purely
to reach a cheaper tier sooner. If the disposal cost is too low, the model games the learning
mechanism, and the result is an artefact.

Two experiments. First, sweep the disposal penalty down to zero.

In [ ]:
rows = []
for pen in [12.0, 6.0, 3.0, 1.0, 0.0]:
    m = build_model(learning='production', pen_dispose=pen); m.optimize()
    d = sum(m._v['disp'][r, p].X for r in REGIONS for p in P)
    rows.append(dict(disposal_penalty=pen, objective=round(m.ObjVal, 1),
                     disposal_units=round(d, 2)))
pd.DataFrame(rows)

Zero disposal at every level, **including free disposal**. Second experiment: make learning
implausibly strong and try again.

In [ ]:
saved_lr, saved_floor = LR_OPEX, OPEX_FLOOR
rows = []
for lr, fl in [(0.18, 0.65), (0.35, 0.25), (0.55, 0.25)]:
    globals()['LR_OPEX'], globals()['OPEX_FLOOR'] = lr, fl
    set_tiers(prod)
    m = build_model(learning='production', pen_dispose=0.0); m.optimize()
    d = sum(m._v['disp'][r, p].X for r in REGIONS for p in P)
    rows.append(dict(LR_opex=lr, multipliers=[round(x, 3) for x in TIER_M['PROC']],
                     disposal_units=round(d, 2), objective=round(m.ObjVal, 1)))
globals()['LR_OPEX'], globals()['OPEX_FLOOR'] = saved_lr, saved_floor
set_tiers(prod)
pd.DataFrame(rows)

Still zero, even with a 55% learning rate and free disposal.

**The binding consideration is not the disposal penalty at all.** Utilization is already 90–97%,
so there is no spare capacity to overproduce with — dumping would require *building* a facility.
Under cost minimization that means paying capex, opex and transport to obtain an operating-cost
discount, with **no revenue upside**. It never pays.

So $\pi^{disp} = 12$ is safely above the threshold, and the threshold is essentially unreachable
in a cost-minimizing model. This confirms the expectation that predatory overproduction is a
**competitive** phenomenon: it needs revenue and market share to be rational, which arrives in
Part 4. The mechanism is verified here and idle — exactly the right state for scaffolding.

## 14. The government lever: local content minimums

A minimum throughput at each stage and region is a **local content requirement** — a live policy
instrument in critical minerals. Firms then decide how to meet it.

Two design points. It is **one-sided**: exceeding the minimum is not a violation, so only
undersupply is penalised. And it **phases in**, because a minimum that binds before any asset could
plausibly be online would produce a violation that is a lead-time artefact rather than a policy
result.

Because deviation is a variable with a penalty rather than a hard constraint, an unachievable
policy shows up as a *reportable quantity* instead of an infeasible model.

In [ ]:
rows = []
for level in [0.0, 60.0, 110.0, 160.0]:
    set_tier_minimums(level)
    m = build_model(learning='production'); m.optimize()
    v = m._v
    rows.append(dict(min_throughput=level, objective=round(m.ObjVal, 1),
                     undersupply=round(sum(v['dev'][s, r, p].X
                                           for (s, r) in NODES for p in P), 2),
                     disposal=round(sum(v['disp'][r, p].X
                                        for r in REGIONS for p in P), 2),
                     builds=sum(1 for k in BUILD if v['build'][k].X > 0.5)))
set_tier_minimums(0.0)
pd.DataFrame(rows)

This is where the disposal mechanism finally activates, and it validates the wiring.

At a minimum of 160 the policy forces more throughput than final demand can absorb. The model
complies — undersupply stays at zero — by building **two extra facilities** and dumping the
surplus. The objective jumps sharply. That is the real cost of an aggressive local content rule,
and it is visible as three separate quantities (extra capex, extra opex, disposal) rather than one
opaque number.

Note the ordering that makes this behaviour meaningful: the undersupply penalty (35) exceeds the
disposal cost (12), so complying-and-dumping is cheaper than violating. Reverse that ordering and
the model would rationally violate the quota instead. **The relative sizes of the three penalties
determine behaviour more than their absolute values**, so they should be stated and swept, not
picked once.

## 15. Does production-based learning change the *plan*?

In [ ]:
rows = []
for mode in ['none', 'capacity', 'production', 'both']:
    m = build_model(learning=mode); m.optimize(); v = m._v
    plan = sorted((START[vv], s, r, round(v['size'][s, r, vv].X, 1))
                  for (s, r, vv) in BUILD if v['build'][s, r, vv].X > 0.5)
    rows.append(dict(learning=mode, n_builds=len(plan),
                     total_capacity=round(sum(x[3] for x in plan), 1),
                     mean_size=round(sum(x[3] for x in plan)/max(1, len(plan)), 1),
                     build_years=[x[0] for x in plan]))
pd.DataFrame(rows)

**The plan is essentially unchanged.** Same number of builds, same years, same total capacity.
Only the cost differs.

This is a real result, and it is worth stating plainly rather than dressing up: in a
**demand-pulled, cost-minimizing** model, production-based learning is very nearly a *windfall
rather than a decision driver*. Cumulative production is pinned by demand — the model must serve
it, cannot profitably exceed it, and has little freedom over when. So the tiers arrive on a
schedule the model can barely influence, and the investment plan does not move.

Two conditions would make it a genuine driver, and both are Part 4:

1. **Revenue and price.** Once output can be sold rather than merely delivered, producing more has
   an upside, and buying down the curve becomes a strategy.
2. **Rivalry with regional scope.** If learning stays inside a region, reaching a tier first is a
   durable cost advantage over a competitor — which is exactly the mechanism behind flooding a
   market to starve an entrant.

So the honest reading of Part 3b is: the *formulation* is now correct and the *mechanism* is
verified, but the interesting dynamics need the competitive setting. Getting the machinery right
here — where behaviour is predictable and every quantity can be checked against a hand argument —
is what makes Part 4 trustworthy.

## 16. Summary and what carries forward

### Findings

| Question | Answer |
|---|---|
| Do the two channels separate cleanly? | Yes — A moves capex, B moves opex |
| Is capacity-based learning a valid bound? | Yes, and the gap is measurable |
| Is utilization high, as predicted? | Yes, 76–97%, weighted by capacity-years |
| Do the tiers activate? | Yes, monotone 0→1→2, with the lag visible |
| Is pump-and-dump profitable? | **No** — not even with free disposal and LR = 55% |
| Does production learning change the plan? | **No** — under cost minimization it is a windfall |
| Does the disposal mechanism work at all? | Yes — an aggressive local content rule triggers it |

### Formulation lessons

- **Physical accumulation is undiscounted.** $L_p$ for cumulative production, $\omega_p$ for money.
- **Define lags in years**, then map to periods, or learning decelerates as periods coarsen.
- **Calibrate thresholds from a solve.** A threshold outside the realised range is a dead binary.
- **Aggregate where the rate is constant.** Opex does not depend on vintage, so splitting
  *node-level* throughput across tiers is exactly equivalent to splitting vintage-level — and it
  cut roughly 2,200 variables, which is what brought the model back under the licence cap.
- **Penalty ordering is a modelling decision.** Undersupply > disposal makes compliance cheaper
  than violation. Reverse it and the model games the policy.

### Carrying into Part 4

Everything the competitive game needs is now built and tested: cumulative production state,
production-driven learning with regional or global scope, overproduction with a disposal cost, and
local content minimums as an exogenous government lever. What is missing is only the strategic
layer — endogenous price, per-region profit objectives, and an equilibrium concept.

### Things to try

- `LEARN_SCOPE = 'global'` then re-calibrate — spillover slightly lowers cost and weakens any
  first-mover advantage
- `LAG_YEARS = 0` — instantaneous learning; tiers arrive sooner and the objective falls
- `N_TIERS = 5` — a finer step approximation, at the cost of more binaries; watch for bunching
- `PEN_DEVIATE = 5` with `min_throughput = 160` — now violating is cheaper than complying, and the
  model should abandon the quota